# Giai đoạn 3 (Tiếp nối): Resume Training ExDark +30 Epochs
Notebook này tiếp tục quá trình huấn luyện từ **Epoch 50** (file `last.pt`).
- Model vẫn đang trong xu hướng cải thiện (chưa bão hòa) → quyết định vắt kiệt tiềm năng.
- Tổng cộng sau khi hoàn tất: **80 Epochs** (50 cũ + 30 mới).
- Tích hợp bộ đếm thời gian tổng để theo dõi chi phí tính toán.

In [1]:
# Kiểm tra GPU & VRAM
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.2f} GB")
else:
    print("CẢNH BÁO: Không phát hiện CUDA GPU!")

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.29 GB


## Lưu Mốc Lịch Sử (Checkpoint Milestone - Epoch 50)
Trước khi Resume ghi đè, sao lưu toàn bộ trọng số và bảng số liệu của Pha 1 (50 Epochs) thành bản đánh dấu mốc.
- `last.pt` → `last_epoch50.pt` (trọng số tại thời điểm dừng)
- `best.pt` → `best_epoch50.pt` (trọng số tốt nhất trong 50 epoch đầu)
- `results.csv` → `results_epoch50.csv` (toàn bộ log huấn luyện Pha 1)
- Các biểu đồ → lưu vào thư mục con `milestone_epoch50/`

In [2]:
import shutil, os

run_dir = 'd:/DAT301m/proposal/models/runs/exdark_train-2'
milestone_dir = os.path.join(run_dir, 'milestone_epoch50')
os.makedirs(milestone_dir, exist_ok=True)

# Sao lưu trọng số
shutil.copy2(f'{run_dir}/weights/last.pt', f'{run_dir}/weights/last_epoch50.pt')
shutil.copy2(f'{run_dir}/weights/best.pt', f'{run_dir}/weights/best_epoch50.pt')

# Sao lưu biểu đồ & ma trận
for fname in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
              'BoxPR_curve.png', 'BoxF1_curve.png', 'BoxP_curve.png', 'BoxR_curve.png']:
    src = os.path.join(run_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(milestone_dir, fname))

print(f'Checkpoint Milestone Epoch 50 saved to: {milestone_dir}')
print(f'Weights backup: last_epoch50.pt, best_epoch50.pt')
print(f'Files saved: {os.listdir(milestone_dir)}')

Checkpoint Milestone Epoch 50 saved to: d:/DAT301m/proposal/models/runs/exdark_train-2\milestone_epoch50
Weights backup: last_epoch50.pt, best_epoch50.pt
Files saved: ['BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'results.png', 'results_epoch50.csv']


## Resume Training từ last.pt
- Nạp trọng số từ Epoch 50 (`last.pt`) thay vì `yolo11n.pt` (trọng số COCO gốc).
- Tổng epochs = **80** (hệ thống tự biết đã train 50 rồi, sẽ chỉ chạy thêm 30 epoch nữa).
- Giữ nguyên toàn bộ siêu tham số để đảm bảo tính nhất quán thí nghiệm.
- Kèm bộ đếm thời gian tổng (phút).

In [3]:
import time
from ultralytics import YOLO

start_time = time.time()

# Nạp trọng số từ lần train trước (Epoch 50)
model = YOLO('d:/DAT301m/proposal/models/runs/exdark_train-2/weights/last.pt')

# Resume Training thêm 30 Epochs (tổng = 80)
results = model.train(
    data='d:/DAT301m/proposal/data/exdark.yaml',
    epochs=80,              # Tổng epochs mong muốn (hệ thống tự tính: 80 - 50 = 30 epochs còn lại)
    patience=15,            # Dừng sớm nếu sau 15 epoch mAP không cải thiện
    batch=8,                # Giữ nguyên Batch Size
    cache='disk',           # Bộ đệm ổ cứng giảm tải System RAM
    imgsz=640,
    device=0,
    optimizer='AdamW',
    mosaic=0.5,
    close_mosaic=10,        # Tắt Mosaic ở 10 epoch cuối (epoch 70-80)
    resume=True,            # BẮT BUỘC: Kích hoạt cơ chế Resume
    project='d:/DAT301m/proposal/models/runs',
    name='exdark_train-2',
    exist_ok=True           # Cho phép ghi tiếp vào thư mục cũ
)

elapsed = time.time() - start_time
print(f"HOÀN TẤT RESUME TRAINING")
print(f"Thời gian pha Resume: {elapsed/60:.1f} phút ({elapsed:.0f} giây)")
print(f"Tổng thời gian tích lũy (cả 2 pha): {(7230 + elapsed)/60:.1f} phút")

Ultralytics 8.4.76  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\DAT301m\proposal\data\exdark.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=d:\DAT301m\proposal\models\runs\exdark_train-2\weights\last.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=exdark_train-2, nbs=64, nm